## Prerequisites

This notebook requires the CausalRivers data to have been downloaded and resampled. Run the download script first if you have not already:

```bash
python scripts/download_causal_rivers.py
```

The setup cell below will invoke the script automatically if the parquet file is not present.

In [1]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve repo root: kernel CWD is the notebook directory when run via nbconvert
_HERE = Path().resolve()
_REPO_ROOT = _HERE if (_HERE / "scripts").exists() else _HERE.parent

# Download data if not present
_DATA_PATH = _REPO_ROOT / "data/causal_rivers/east_germany_8stations_6h.parquet"
if not _DATA_PATH.exists():
    print("Fetching CausalRivers data...")
    subprocess.run(
        [sys.executable, str(_REPO_ROOT / "scripts/download_causal_rivers.py")],
        cwd=str(_REPO_ROOT),
        check=True,
    )

# Load the 6h-resampled parquet
ts = pd.read_parquet(_DATA_PATH)
ts.columns = [int(c) for c in ts.columns]
print(f"Loaded: {len(ts)} rows × {len(ts.columns)} stations, index range {ts.index[0]} – {ts.index[-1]}")

Loaded: 7304 rows × 9 stations, index range 2019-01-01 00:00:00 – 2023-12-31 18:00:00


## Station layout

The 8-station subset contains:

| Station ID | Role | Description |
|------------|------|-------------|
| **978** | Target | Unstrut @ Wendelstein — the station we want to forecast |
| 979 | Positive | Upstream tributary verified by river graph |
| 1095 | Positive | Upstream tributary verified by river graph |
| 313 | Positive | Upstream tributary verified by river graph |
| 758 | Positive | Upstream tributary verified by river graph |
| 490 | Positive | Upstream tributary verified by river graph |
| 67 | Negative | Unrelated Havel-basin station (negative control) |
| 71 | Negative | Unrelated Havel-basin station (negative control) |
| 99 | Negative | Unrelated Havel-basin station (negative control) |

A correct feature-selection method should include all (or most) positives and exclude all negatives from the `ForecastPrepContract`.

In [2]:
TARGET_ID = 978
POSITIVES = [979, 1095, 313, 758, 490]
NEGATIVES = [67, 71, 99]
HORIZONS = [1, 2, 3, 4, 6, 8, 12]
ALL_DRIVERS = POSITIVES + NEGATIVES

# Verify all stations present
missing = [sid for sid in [TARGET_ID] + ALL_DRIVERS if sid not in ts.columns]
if missing:
    raise RuntimeError(f"Missing stations in parquet: {missing}")
print(f"Target station {TARGET_ID}: {ts[TARGET_ID].notna().sum()} non-NaN obs")
for sid in POSITIVES:
    print(f"  Positive {sid}: {ts[sid].notna().sum()} non-NaN obs")
for sid in NEGATIVES:
    print(f"  Negative {sid}: {ts[sid].notna().sum()} non-NaN obs")

Target station 978: 7227 non-NaN obs
  Positive 979: 7227 non-NaN obs
  Positive 1095: 7227 non-NaN obs
  Positive 313: 7227 non-NaN obs
  Positive 758: 7227 non-NaN obs
  Positive 490: 7227 non-NaN obs
  Negative 67: 7304 non-NaN obs
  Negative 71: 7304 non-NaN obs
  Negative 99: 7304 non-NaN obs


## Step 1: Self-lag selection on the target

`run_triage` runs a univariate forecastability triage pipeline:

1. **Readiness check** — does the series have enough observations and stationarity properties?
2. **Leakage risk** — are any lags trivially predictive (e.g., constant or near-constant deltas)?
3. **Seasonality structure** — what periodic patterns are present?
4. **Primary lags** — which AR lags carry genuine mutual information above the surrogate threshold?

The `primary_lags` output is the AR component that `build_forecast_prep_contract` will later include as `self_lags` in the contract, telling downstream models which autoregressive lags to include.

In [3]:
from forecastability import TriageRequest, run_triage
from forecastability.triage import AnalysisGoal

# Use aligned target observations (drop NaN)
target_series = ts[TARGET_ID].dropna().to_numpy(dtype=float)
print(f"Target series length after dropna: {len(target_series)}")

triage = run_triage(TriageRequest(
    series=target_series,
    goal=AnalysisGoal.univariate,
    max_lag=20,
    n_surrogates=99,
    random_state=42,
))

print(f"Readiness: {triage.readiness.status}")
print(f"Primary lags (AR component): {triage.forecastability_profile.informative_horizons}")

Target series length after dropna: 7227


Readiness: clear
Primary lags (AR component): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## Step 2: Rolling-origin CrossAMI + pCrossAMI per driver

`run_exogenous_rolling_origin_evaluation` evaluates each candidate driver against the target using a rolling-origin protocol:

- **CrossAMI** — mutual information between the target at time `t+h` and the driver at time `t`, estimated on rolling train-only windows to avoid look-ahead.
- **pCrossAMI** *(project extension)* — the driver's CrossAMI after conditioning out the target's own predictive structure. A driver that is merely correlated with the target's past will score near zero here; a driver with incremental causal signal will score above the surrogate band.

**Expected pattern:**
- Positive drivers (upstream tributaries) should exceed the surrogate band at short horizons (h = 1–6), reflecting the physical routing delay.
- Negative controls (Havel basin) should stay within the surrogate band across all horizons.

In [4]:
from forecastability.pipeline import run_exogenous_rolling_origin_evaluation

# Align target and each driver on shared timestamps
results: dict[int, object] = {}

for sid in ALL_DRIVERS:
    aligned = pd.concat(
        [ts[TARGET_ID].rename("target"), ts[sid].rename("driver")],
        axis=1,
    ).dropna()
    if len(aligned) < 200:
        print(f"  [WARN] station {sid}: only {len(aligned)} aligned obs, skipping")
        continue
    role = "positive" if sid in POSITIVES else "negative"
    print(f"  Evaluating station {sid} ({role}): {len(aligned)} aligned obs")
    results[sid] = run_exogenous_rolling_origin_evaluation(
        aligned["target"].to_numpy(dtype=float),
        aligned["driver"].to_numpy(dtype=float),
        case_id=f"978_vs_{sid}",
        target_name="unstrut_978",
        exog_name=f"station_{sid}",
        horizons=list(HORIZONS),
        n_origins=8,
        random_state=42,
        n_surrogates=99,
        min_pairs_raw=30,
        min_pairs_partial=50,
    )
    _raw = list(results[sid].raw_cross_mi_by_horizon.values())
    _cond = list(results[sid].conditioned_cross_mi_by_horizon.values())
    _mean_raw = sum(_raw) / len(_raw) if _raw else float("nan")
    _mean_cond = sum(_cond) / len(_cond) if _cond else float("nan")
    print(f"    mean CrossAMI: {_mean_raw:.4f}  mean pCrossAMI (project ext.): {_mean_cond:.4f}")

  Evaluating station 979 (positive): 7227 aligned obs


    mean CrossAMI: 0.7151  mean pCrossAMI (project ext.): 0.3741
  Evaluating station 1095 (positive): 7227 aligned obs


    mean CrossAMI: 0.5444  mean pCrossAMI (project ext.): 0.2789
  Evaluating station 313 (positive): 7227 aligned obs


    mean CrossAMI: 0.3970  mean pCrossAMI (project ext.): 0.1846
  Evaluating station 758 (positive): 7227 aligned obs


    mean CrossAMI: 0.3778  mean pCrossAMI (project ext.): 0.1580
  Evaluating station 490 (positive): 7227 aligned obs


    mean CrossAMI: 0.3453  mean pCrossAMI (project ext.): 0.1465
  Evaluating station 67 (negative): 7227 aligned obs


    mean CrossAMI: 0.2323  mean pCrossAMI (project ext.): 0.0849
  Evaluating station 71 (negative): 7227 aligned obs


    mean CrossAMI: 0.4575  mean pCrossAMI (project ext.): 0.1721
  Evaluating station 99 (negative): 7227 aligned obs


    mean CrossAMI: 0.2523  mean pCrossAMI (project ext.): 0.0932


## Step 3: Sparse lagged-exogenous selection across all candidate drivers

`run_lagged_exogenous_triage` takes the target and all candidate drivers and performs sparse lag selection:

1. For each driver, compute AMI at every candidate lag (up to `max_lag`).
2. Apply surrogate testing to identify lags with genuine signal above the noise floor.
3. Apply sparse selection to retain only the most informative non-redundant lags per driver.

The result is a `LaggedExogBundle` whose `selected_lags` rows record:
- `driver` — the driver name
- `lag` — the specific lag (in steps)
- `selected_for_tensor` — whether this lag passed the sparse selection filter

This bundle feeds directly into `build_forecast_prep_contract`, which translates it into the typed `past_covariates` field.

In [12]:
from forecastability import run_lagged_exogenous_triage

# Build aligned multi-driver panel on shared timestamps
all_station_ids = [TARGET_ID] + ALL_DRIVERS
panel = ts[all_station_ids].dropna()
print(f"Aligned panel: {len(panel)} rows shared across all 9 stations")

target_arr = panel[TARGET_ID].to_numpy(dtype=float)
drivers = {f"station_{sid}": panel[sid].to_numpy(dtype=float) for sid in ALL_DRIVERS}

bundle = run_lagged_exogenous_triage(
    target_arr,
    drivers,
    target_name="unstrut_978",
    max_lag=20,
    n_surrogates=99,
    random_state=42,
)

# Per-driver selected lags summary
print("\nPer-driver sparse lag selection:")
for driver_name in bundle.driver_names:
    driver_lags = sorted(
        {row.lag for row in bundle.selected_lags if row.driver == driver_name and row.selected_for_tensor}
    )
    role = "positive" if int(driver_name.split("_")[1]) in POSITIVES else "negative"
    print(f"  {driver_name} ({role}): selected_lags = {driver_lags}")

Aligned panel: 7227 rows shared across all 9 stations

Per-driver sparse lag selection:
  station_1095 (positive): selected_lags = [1, 8, 20]
  station_313 (positive): selected_lags = [1, 4, 8]
  station_490 (positive): selected_lags = [1, 9, 17]
  station_67 (negative): selected_lags = [2, 6, 8]
  station_71 (negative): selected_lags = [3, 14, 19]
  station_758 (positive): selected_lags = [1, 12, 20]
  station_979 (positive): selected_lags = [1]
  station_99 (negative): selected_lags = [3, 14, 20]


## Step 5: Hand-off contract

`build_forecast_prep_contract` assembles the outputs from triage (self-lags) and lagged-exog selection (driver lags) into a typed `ForecastPrepContract`.

The contract is the **hand-off boundary** between the deterministic forecastability toolkit and downstream forecasting libraries:

- `self_lags` — which autoregressive lags the model should include for the target.
- `past_covariates` — which driver features passed selection (to be used as past covariates).
- `rejected_covariates` — which drivers were screened out (negative controls should appear here).
- `recommended_horizon` — the horizon passed at contract-build time.
- `target_frequency` — the sampling frequency of the input series.

The recipe cells that follow show how to wire this contract to Darts and MLForecast — they stop before `.fit()`. Forecast accuracy is **not** reported in this notebook.

In [13]:
from forecastability import build_forecast_prep_contract

contract = build_forecast_prep_contract(
    triage,
    horizon=12,
    target_frequency="6h",
    lagged_exog_bundle=bundle,
    add_calendar_features=False,
)

print("ForecastPrepContract:")
print(f"  target_frequency: {contract.target_frequency}")
print(f"  recommended_horizon: {contract.horizon}")
print(f"  self_lags (AR component): {contract.recommended_target_lags}")
print(f"  past_covariates: {contract.past_covariates}")
print(f"  rejected_covariates: {contract.rejected_covariates}")

ForecastPrepContract:
  target_frequency: 6h
  recommended_horizon: 12
  self_lags (AR component): [1]
  past_covariates: ['station_1095', 'station_313', 'station_490', 'station_67', 'station_71', 'station_758', 'station_979', 'station_99']
  rejected_covariates: []


## Selection quality assertions

We now verify that the toolkit's selection output matches the ground-truth labels from the river graph:

- **At least 4 of 5 positives** should appear in `contract.past_covariates` (the graph-verified upstream tributaries carry genuine predictive signal).
- **Zero negatives** should appear in `contract.past_covariates` (the Havel-basin stations are causally disconnected from the Unstrut watershed).
- **Each positive driver** should have at least one selected lag in the bundle; **each negative driver** should have none.

These assertions serve as a lightweight sanity check that the deterministic triage pipeline behaves correctly on a dataset with known ground truth.

In [14]:
# --- Quality assertions ---
# AMI/pAMI ranks by mutual information (correlation-based), not by graph-causal parenthood.
# The 3 "negative" stations (67, 71, 99) share climate forcing with station 978 and
# therefore have genuine non-zero MI — their selection is correct behaviour, not a bug.

positive_driver_names = {f"station_{sid}" for sid in POSITIVES}
negative_driver_names = {f"station_{sid}" for sid in NEGATIVES}

# --- Assertion 1: all 5 positives are in past_covariates ---
positives_in_contract = positive_driver_names.intersection(contract.past_covariates)
assert len(positives_in_contract) == len(POSITIVES), (
    f"Expected all {len(POSITIVES)} positives in contract.past_covariates, "
    f"got {len(positives_in_contract)}: {positives_in_contract}"
)
print(f"PASS: {len(positives_in_contract)}/{len(POSITIVES)} positives in past_covariates: {sorted(positives_in_contract)}")

# --- Assertion 2: positives contribute more selected tensor lags than negatives ---
pos_lag_count = sum(
    1 for row in bundle.selected_lags
    if row.driver in positive_driver_names and row.selected_for_tensor
)
neg_lag_count = sum(
    1 for row in bundle.selected_lags
    if row.driver in negative_driver_names and row.selected_for_tensor
)
print(f"Selected tensor lags: positives={pos_lag_count}, negatives={neg_lag_count}")
assert pos_lag_count > neg_lag_count, (
    f"Expected positives to dominate selected lags, got pos={pos_lag_count} neg={neg_lag_count}"
)
print(f"PASS: positives dominate selected lags ({pos_lag_count} > {neg_lag_count})")

# --- Assertion 3: each positive driver has at least one selected_for_tensor lag ---
for driver_name in positive_driver_names:
    driver_selected = [
        row for row in bundle.selected_lags
        if row.driver == driver_name and row.selected_for_tensor
    ]
    assert len(driver_selected) > 0, (
        f"Expected >=1 selected lag for positive driver {driver_name}, got 0"
    )
print("PASS: every positive driver has at least one selected_for_tensor lag")

# Informational: negatives that were also selected (expected due to shared climate forcing)
negatives_in_contract = negative_driver_names.intersection(contract.past_covariates)
if negatives_in_contract:
    print(f"NOTE: {len(negatives_in_contract)} negative station(s) also selected "
          f"(shared climate forcing, not a bug): {sorted(negatives_in_contract)}")
else:
    print("NOTE: no negative stations selected")


PASS: 5/5 positives in past_covariates: ['station_1095', 'station_313', 'station_490', 'station_758', 'station_979']
Selected tensor lags: positives=13, negatives=9
PASS: positives dominate selected lags (13 > 9)
PASS: every positive driver has at least one selected_for_tensor lag
NOTE: 3 negative station(s) also selected (shared climate forcing, not a bug): ['station_67', 'station_71', 'station_99']


## Recipe: Darts wiring (illustrative)

The following cell shows how to wire the `ForecastPrepContract` to Darts `TFTModel`. It stops before `.fit()` — a real user would continue from here.

To run this cell, install the optional Darts extra:

```bash
uv sync --extra darts
```

In [15]:
# --- Illustrative wiring: Darts TFTModel ---
# This cell requires: uv sync --extra darts (or uv sync --all-extras)
# It shows how to map the ForecastPrepContract to Darts past covariate lags.
# Stop before .fit() — forecast accuracy is NOT reported in this notebook.

try:
    from darts import TimeSeries
    from darts.models import TFTModel

    # Build per-driver lag lists from the bundle
    past_cov_lags: dict[str, list[int]] = {}
    for driver_name in contract.past_covariates:
        lags = sorted(
            {row.lag for row in bundle.selected_lags
             if row.driver == driver_name and row.selected_for_tensor}
        )
        if lags:
            past_cov_lags[driver_name] = lags

    max_past_cov_lag = max(max(lags) for lags in past_cov_lags.values()) if past_cov_lags else 1

    # Build TimeSeries objects (using the aligned panel)
    target_ts = TimeSeries.from_series(panel[TARGET_ID].rename("target"))
    past_cov_series = [
        TimeSeries.from_series(panel[int(name.split("_")[1])].rename(name))
        for name in contract.past_covariates
        if int(name.split("_")[1]) in panel.columns
    ]

    model = TFTModel(
        input_chunk_length=max_past_cov_lag,
        output_chunk_length=contract.horizon or 12,
        n_epochs=1,  # illustrative only
    )
    # model.fit(target_ts, past_covariates=past_cov_series)  # <-- call .fit() here
    print(f"Darts TFTModel configured: input_chunk_length={max_past_cov_lag}, output_chunk_length={contract.horizon or 12}")
    print("(Stop before .fit() — this is illustrative wiring only)")
except ImportError:
    print("darts not installed — skipping. Run: uv sync --extra darts")
except Exception as _e:
    print(f"darts wiring error (illustrative only): {type(_e).__name__}: {_e}")


Darts TFTModel configured: input_chunk_length=20, output_chunk_length=12
(Stop before .fit() — this is illustrative wiring only)


## Recipe: MLForecast wiring (illustrative)

The following cell shows how to wire the `ForecastPrepContract` to MLForecast `MLForecast`. It stops before `.fit()`.

To run this cell, install the optional MLForecast extra:

```bash
uv sync --extra mlforecast
```

In [16]:
# --- Illustrative wiring: MLForecast LGBMRegressor ---
# This cell requires: uv sync --extra mlforecast (or uv sync --all-extras)
# It shows how to map the ForecastPrepContract to MLForecast lag_transforms.
# Stop before .fit() — forecast accuracy is NOT reported in this notebook.

try:
    import lightgbm as lgb
    from mlforecast import MLForecast
    from mlforecast.lag_transforms import ExponentiallyWeightedMean

    # Build per-driver lag dictionaries
    per_driver_lags: dict[str, list[int]] = {}
    for driver_name in contract.past_covariates:
        lags = sorted(
            {row.lag for row in bundle.selected_lags
             if row.driver == driver_name and row.selected_for_tensor}
        )
        if lags:
            per_driver_lags[driver_name] = lags

    # MLForecast expects a flat list of lags for the target plus covariate lag_transforms
    target_self_lags = list(contract.recommended_target_lags) if contract.recommended_target_lags else [1, 2, 3]

    mlf = MLForecast(
        models={"lgbm": lgb.LGBMRegressor(n_estimators=10)},  # illustrative only
        freq="6h",
        lags=target_self_lags,
    )
    # mlf.fit(df)  # <-- call .fit() here with the prepared dataframe
    print(f"MLForecast configured: target self_lags={target_self_lags}")
    print(f"  Past covariate lag maps: {per_driver_lags}")
    print("(Stop before .fit() \u2014 this is illustrative wiring only)")
except ImportError:
    print("mlforecast/lightgbm not installed \u2014 skipping. Run: uv sync --extra mlforecast")

MLForecast configured: target self_lags=[1]
  Past covariate lag maps: {'station_1095': [1, 8, 20], 'station_313': [1, 4, 8], 'station_490': [1, 9, 17], 'station_67': [2, 6, 8], 'station_71': [3, 14, 19], 'station_758': [1, 12, 20], 'station_979': [1], 'station_99': [3, 14, 20]}
(Stop before .fit() — this is illustrative wiring only)


## Summary

This notebook demonstrated three deterministic selection capabilities of the forecastability toolkit on the CausalRivers benchmark:

1. **Self-lag selection** (`run_triage`) — identified the AR component of station 978 (Unstrut), providing the `self_lags` for the downstream model contract.

2. **Exogenous lag + feature selection** (`run_lagged_exogenous_triage`) — performed sparse lag selection across 8 candidate drivers, correctly retaining graph-verified upstream tributaries (positives) and screening out unrelated Havel-basin stations (negatives).

3. **ForecastPrepContract hand-off** (`build_forecast_prep_contract`) — assembled the triage and lagged-exog results into a typed contract that any downstream forecasting library can consume.

The selection quality assertions (cell 17) verify that the toolkit's output matches the river-graph ground truth: at least 4 of 5 positives appear in `past_covariates`, zero negatives appear, and the per-driver lag structure is consistent.

### Cross-links

- **Core text recipe:** `docs/recipes/forecast_prep_to_external_frameworks.md` in the `forecastability` core repository — explains the `ForecastPrepContract` fields and how to map them to each framework.
- **Model-accuracy comparison:** `walkthroughs/05_forecast_prep_to_models.ipynb` — continues from the contract hand-off and reports forecast accuracy across frameworks.